# Hands-on with Kafka

![kafka](https://images.contentful.com/gt6dp23g0g38/53UO4964r0e7kRVm0mcUUZ/f6f6d7b1b90e8e88a5be0d1845bdf950/what_is_kafka_and_how_does_it_work.png)

## Installing Kafka Python Client

First, we'll need to install Kafka Python client `kafka-python` to consume messages from a Kafka cluster.

In [1]:
import json
from kafka import KafkaConsumer, TopicPartition

First, we'll instantiate the consumer.

Our data was serialized in JSON during publishing, hence we'll need to deserialize as JSON when consuming. In actual production, it is recommended to use the binary format that we've learnt, such as Avro.

In [2]:
consumer = KafkaConsumer(
  bootstrap_servers=['localhost:9092'],
  auto_offset_reset='earliest',
  value_deserializer = lambda v: json.loads(v.decode('ascii')),
  key_deserializer = lambda v: json.loads(v.decode('ascii')),
)

We set the offset for the consumer to be `earliest` to get the earliest order (message).

List the available topics:

In [3]:
consumer.topics()

{'pizza-orders'}

Let's assume we are the owners of a pizza delivery chain, and we'd like to push the orders to Apache Kafka as they come in.

We have a topic consisting of pizza orders, which contains the Order ID, Order Details (pizzas and toppings), client's Name, Address, Phone Number, the Shop Name, Total Cost and Timestamp.

There is only one partition for the topic. First, let's subscribe to the topic:

In [4]:
topic_name = "pizza-orders"

In [5]:
consumer.subscribe(topics=[topic_name])
consumer.subscription()

{'pizza-orders'}

We can then start reading the events, let's read and (pretty) print the first 5:

In [6]:
from pprint import pprint

In [7]:
for ix, message in enumerate(consumer, start=1):
    print("%d:%d: k=%s" % (message.partition,
                           message.offset,
                           message.key))
    pprint("-------------------")
    pprint(message.value)
    print()
    if ix == 5:
        break

0:0: k={'shop': 'Luigis Pizza'}
'-------------------'
{'address': '113 Reeves Oval\nChanbury, UT 40179',
 'cost': 39.72,
 'id': 0,
 'name': 'Katherine Wright',
 'phoneNumber': '922-262-0398x5836',
 'pizzas': [{'additionalToppings': ['🍅 tomato', '🥓 bacon'],
             'pizzaName': 'Peperoni'},
            {'additionalToppings': ['🐟 tuna', '🍓 strawberry', '🐟 tuna'],
             'pizzaName': 'Mari & Monti'},
            {'additionalToppings': ['🐟 tuna', '🥚 egg'],
             'pizzaName': 'Mari & Monti'},
            {'additionalToppings': ['🌶️ hot pepper'], 'pizzaName': 'Diavola'}],
 'shop': 'Luigis Pizza',
 'timestamp': 1780628823295}

0:1: k={'shop': 'Ill Make You a Pizza You Cant Refuse'}
'-------------------'
{'address': '246 Ramirez Island Apt. 173\nWrightland, KY 20970',
 'cost': 0.71,
 'id': 1,
 'name': 'Erin Roth',
 'phoneNumber': '671.340.8222x66282',
 'pizzas': [{'additionalToppings': ['🧅 onion', '🍅 tomato'],
             'pizzaName': 'Marinara'},
            {'additionalTop

Now, continue printing the next 5 events, notice the offset number and order id:

In [8]:
for ix, message in enumerate(consumer, start=1):
    print("%d:%d: k=%s" % (message.partition,
                           message.offset,
                           message.key))
    pprint(message.value)
    print()
    if ix == 5:
        break

0:5: k={'shop': 'Ill Make You a Pizza You Cant Refuse'}
{'address': '594 Stephanie Common Suite 958\nLake Lauriemouth, NC 24445',
 'cost': 1.46,
 'id': 0,
 'name': 'Mathew Collins',
 'phoneNumber': '9452932049',
 'pizzas': [{'additionalToppings': ['🌶️ hot pepper', '🧄 garlic', '🍌 banana'],
             'pizzaName': 'Salami'},
            {'additionalToppings': ['🐟 tuna', '🐟 tuna', '🫒 olives'],
             'pizzaName': 'Marinara'},
            {'additionalToppings': [], 'pizzaName': 'Diavola'}],
 'shop': 'Ill Make You a Pizza You Cant Refuse',
 'timestamp': 1780628952209}

0:6: k={'shop': 'Circular Pi Pizzeria'}
{'address': '97905 Brittany Forge\nPort Elizabethside, HI 75657',
 'cost': 9.89,
 'id': 1,
 'name': 'Justin Holmes',
 'phoneNumber': '+1-798-983-8161x5821',
 'pizzas': [{'additionalToppings': [], 'pizzaName': 'Peperoni'},
            {'additionalToppings': ['🧄 garlic', '🍅 tomato'],
             'pizzaName': 'Margherita'}],
 'shop': 'Circular Pi Pizzeria',
 'timestamp': 178062895

In [18]:
# go to beginning 
consumer.seek_to_beginning()

In [ ]:
# close consumer 
consumer.close()

This is the full code to publish an example order:

```python
from kafka import KafkaProducer

producer = KafkaProducer(
  bootstrap_servers=['brave-fish-11463-us1-kafka.upstash.io:9092'],
  sasl_mechanism='SCRAM-SHA-256',
  security_protocol='SASL_SSL',
  sasl_plain_username='YnJhdmUtZmlzaC0xMTQ2MyQSvwXBuLOQsV1W7YffuC8cDaZcA3fKQwakMhnQGgg',
  sasl_plain_password='MDUxNjc4YzEtYzYxNy00NTE1LWEwNWYtMDBhODRlZmE0OGJm',
  value_serializer=lambda v: json.dumps(v).encode('ascii'),
  key_serializer=lambda v: json.dumps(v).encode('ascii')
)

message = {'address': '8697 Anthony Valley\nPort Kellymouth, FL 64221',
 'id': 10,
 'name': 'Patricia Castaneda',
 'phoneNumber': '328-798-9970x51560',
 'pizzas': [{'additionalToppings': ['🍅 tomato'], 'pizzaName': 'Mari & Monti'}],
 'shop': 'Mammamia Pizza',
 'timestamp': 1696609343719}

key = {'shop': 'Mammamia Pizza'}

producer.send(topic_name, key=key, value=message)
```